# Etsy US Sellers & Transactions - Preprocessing v2 (leakage-safe)

Starts from `preprocessing.ipynb` and turns it into a pipeline that is safe to forecast from.
Two targets, both daily counts per category (and the `ALL` total):

| Target | Question | Horizons (days) |
|---|---|---|
| `orders` | How many orders will be placed in the next *H* days? | 7, 30, 60, 90 |
| `new_sellers` | How many new sellers will sign up in the next *H* days? | 7, 30, 60, 90 |

**The one rule everything below follows:** a forecast is made at the end of day *t* (the **origin**).
Every feature of row *t* may only use data from days **≤ t** (or facts about the future that are
already known at *t*, like holidays). The target is what happens **after** *t*: `y_H(t) = count(t+1) + ... + count(t+H)`.

## What was wrong or missing in `preprocessing.ipynb`

| # | Problem | Why it matters | Fix in this notebook |
|---|---|---|---|
| 1 | **Features were defined relative to the target day, not the origin** (`roll_mean_7` at row *t* uses *t-1 ... t-7*, then the claim "rolling features are safe at every horizon"). | For a 30-day forecast made at origin *t*, day *t+29*'s `roll_mean_7` uses days that do not exist yet. Backtest looks great, production fails. | Step 11-12: row = origin, all history features end at the origin (`lag_0` = origin day), targets are forward sums. |
| 2 | **Imputation fitted on all data** (interpolation with `limit_direction="both"`, KNN, KMeans) *before* any split. Also `order_total_usd` was computed *before* imputation, so imputed prices never reached GMV. | Test-period values leak into training rows; GMV silently ignores every imputed order. | Step 8: split dates are fixed *first*; medians are fitted on the training period only; totals recomputed after imputation. |
| 3 | **Outlier thresholds / winsor quantiles computed on all data.** | Same leak: the test period shapes the training data. | Step 8: fitted on training rows only. |
| 4 | **Feature selection on the full data** (Spearman vs target, noise probes, VIF, RFE, XGBoost tuning) including the test period. | Features are chosen using the answers; backtest is optimistic. | Removed from preprocessing. Belongs *inside* the backtest folds in the modelling notebook. |
| 5 | `avg_order_value = gmv(t) / orders(t)` **same-day ratio** stored next to the target. | It *is* the target, re-expressed. | Replaced by a history-only version (`aov_28`, last 28 days up to the origin). |
| 6 | **Spine ends on the last raw date.** Late-arriving rows make the newest days look low. | The model learns "volume drops at the end" and forecasts too low. | Step 6: data maturity - measure ingestion delay, cut at the last *complete* day. |
| 7 | **No definition of a transaction**: cancelled, pending, refunded all counted the same; EUR/CAD/GBP summed as USD. | "The model is wrong" is usually "we counted different things". | Step 0 `Config`: explicit status and currency rules (confirm with finance). |
| 8 | **Bad rows silently dropped or kept** (unparseable dates -> dropped; unknown status -> NaN kept). | Upstream bugs stay invisible. | Step 5: quarantine with a reason; the run stops if too many rows fail. |
| 9 | **Single 90-day hold-out, no purge/embargo**, train and test targets overlap for multi-day horizons. | The last training targets contain the first test days. | Step 13: `train / purged / test / inference` with an embargo gap. |
| 10 | **No leakage test.** | A stray `shift(-1)` or centred window is invisible in code review. | Step 14: truncation test - rebuild features from data cut at *t*, they must be identical. |
| 11 | **No known-future calendar** (Cyber 5, gifting run-ups, holidays). | These are the biggest drivers of volume and are known years ahead. | Step 10: retail calendar + "how many Cyber-5 / gifting / holiday days fall inside the next *H* days". |
| 12 | **No run manifest**, outputs overwrite each other, settings scattered. | A forecast must be traceable to its data and settings. | Step 0 config + Step 15 manifest. |

Kept from v1 (they were good): explicit date formats, state/boolean normalisation, keep-latest de-duplication,
dense calendar spine with explicit zeros, PASS/FAIL gates, missing-value *flags* instead of blind `fillna`.

Dropped on purpose (less code, no loss): ydata-profiling, one-hot dummies on row tables (not used by the
forecasting tables), all model training (moves to the modelling notebook).

## Step 0: Configuration - every setting in one place

Production pipelines are re-run on a schedule. When a forecast looks wrong, the first question is
*"which settings produced it?"*, so nothing is hard-coded further down. The `Config` is frozen (cannot be
changed mid-run) and is written into the run manifest at the end.

**Assumption to confirm with the data owner:** all timestamps are on one clock. If sources differ
(ISO `...Z` rows are UTC, `MM/DD/YYYY HH:MM` rows are local), convert to the business time zone
*before* flooring to a date - a "day" is a business decision, not a technical one.

In [ ]:
import json
import platform
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from pandas.tseries.holiday import USFederalHolidayCalendar


@dataclass(frozen=True)
class Config:
    # --- inputs / outputs ---
    sellers_path: str = "etsy_us_sellers_raw.csv"
    transactions_path: str = "etsy_us_transactions_raw.csv"
    out_dir: Path = Path("outputs/preprocessing_v2")

    # --- business rules: agree these with finance/ops BEFORE modelling ---
    # An order counts as a transaction once it is placed; a later cancellation does not un-place it.
    transaction_statuses: tuple = ("Completed", "Delivered", "Shipped", "Refunded", "Refund - Partial",
                                   "Cancelled", "Pending", "Processing")
    gmv_statuses: tuple = ("Completed", "Delivered", "Shipped", "Refund - Partial")  # revenue-bearing only
    currency: str = "USD"           # other currencies are quarantined (no FX table in this project)

    # --- data quality ---
    # max_quarantine_rate: float = 0.05   # above this the run STOPS: upstream is broken, not noisy
    # maturity_quantile: float = 0.99     # a day is "complete" once 99% of its rows have arrived
    # anomaly_z: float = 4.0              # robust z-score above which a day is flagged

    # --- forecasting design ---
    horizons: tuple = (7, 30, 60, 90)   # y_H = sum of the H days AFTER the origin
    lags: tuple = (0, 1, 7, 14, 28, 364)  # lag 0 = the origin day itself; 364 = same weekday last year
    roll_windows: tuple = (7, 28, 364)  # windows END on the origin day
    test_days: int = 90                 # final hold-out: evaluated once, at the very end
    embargo_days: int = 7               # gap between the last train target and the first test origin


cfg = Config()
cfg.out_dir.mkdir(parents=True, exist_ok=True)
print(json.dumps(asdict(cfg), indent=2, default=str))

## Step 1: Load raw extracts + data contract

Everything is read **as text**: letting a reader guess types silently turns `"00123"` into `123` and
`03/04/2024` into the wrong month. The *data contract* is the list of columns we depend on; if an
upstream team renames one, the run stops here with a clear message instead of failing cryptically later.

In [ ]:
REQUIRED_COLUMNS = {
    "sellers": ["seller_id", "signup_date_raw", "primary_category_raw", "seller_type_raw", "state_raw",
                "account_status_raw", "is_star_seller_raw", "signup_channel_raw", "row_ingested_at"],
    "transactions": ["order_id", "seller_id", "buyer_id", "order_datetime_raw", "category_raw",
                     "item_price_usd", "quantity", "shipping_fee_usd", "currency", "order_status_raw",
                     "row_ingested_at"],
}


def load_raw(path, table):
    df = pd.read_csv(path, dtype=str)
    df.columns = df.columns.str.strip()
    missing = set(REQUIRED_COLUMNS[table]) - set(df.columns)
    if missing:
        raise ValueError(f"{table}: data contract broken - missing columns {sorted(missing)}")
    return df


sellers_raw = load_raw(cfg.sellers_path, "sellers")
transactions_raw = load_raw(cfg.transactions_path, "transactions")
print("sellers", sellers_raw.shape, "| transactions", transactions_raw.shape)

## Step 2: Pre-clean quality gates

Cheap checks on the raw extract before spending effort cleaning it. Unlike v1, a failed gate **stops the run**
(`raise`) - a warning that scrolls past is the same as no check.

In [ ]:
def run_gates(gates, stage):
    # gates = [(name, passed, detail)]. Print every result, then STOP on any failure.
    failed = []
    for name, passed, detail in gates:
        print(f"[{'PASS' if passed else 'FAIL'}] {name}" + (f" - {detail}" if detail else ""))
        if not passed:
            failed.append(name)
    if failed:
        raise RuntimeError(f"{stage}: {len(failed)} gate(s) failed -> fix the data, do not forecast on it")
    print(f"{stage}: all {len(gates)} gates passed\n")


orphan_rate = (~transactions_raw["seller_id"].str.strip()
               .isin(sellers_raw["seller_id"].str.strip())).mean()
run_gates([
    ("sellers_raw non-empty", len(sellers_raw) > 0, f"{len(sellers_raw):,} rows"),
    ("transactions_raw non-empty", len(transactions_raw) > 0, f"{len(transactions_raw):,} rows"),
    ("seller_id never null", sellers_raw["seller_id"].notna().all(), ""),
    ("order_id never null", transactions_raw["order_id"].notna().all(), ""),
    ("orphan orders (unknown seller) below 10%", orphan_rate < 0.10, f"{orphan_rate:.2%}"),
], "PRE-CLEAN")

## Step 3: Standardise - one spelling, one type per column

The same value arrives as `"COMPLETED"`, `" completed"`, `"Complete"`; a state as `"Texas"`, `"tx"`, `" TX "`;
a flag as `Y`/`true`/`1`. Every known variant is mapped to **one** value; anything unknown becomes missing
(and is caught by the quarantine in Step 5, never silently guessed). Dates are parsed with **explicit**
formats, tried one at a time.

In [ ]:
US_STATE_TO_ABBR = {
    "alabama": "AL", "alaska": "AK", "arizona": "AZ", "arkansas": "AR", "california": "CA",
    "colorado": "CO", "connecticut": "CT", "delaware": "DE", "florida": "FL", "georgia": "GA",
    "hawaii": "HI", "idaho": "ID", "illinois": "IL", "indiana": "IN", "iowa": "IA",
    "kansas": "KS", "kentucky": "KY", "louisiana": "LA", "maine": "ME", "maryland": "MD",
    "massachusetts": "MA", "michigan": "MI", "minnesota": "MN", "mississippi": "MS", "missouri": "MO",
    "montana": "MT", "nebraska": "NE", "nevada": "NV", "new hampshire": "NH", "new jersey": "NJ",
    "new mexico": "NM", "new york": "NY", "north carolina": "NC", "north dakota": "ND", "ohio": "OH",
    "oklahoma": "OK", "oregon": "OR", "pennsylvania": "PA", "rhode island": "RI", "south carolina": "SC",
    "south dakota": "SD", "tennessee": "TN", "texas": "TX", "utah": "UT", "vermont": "VT",
    "virginia": "VA", "washington": "WA", "west virginia": "WV", "wisconsin": "WI", "wyoming": "WY",
    "district of columbia": "DC",
}
VALID_STATE_ABBRS = set(US_STATE_TO_ABBR.values())
MISSING_SENTINELS = {"", "n/a", "na", "none", "null", "unknown"}

ORDER_STATUS_MAP = {
    "completed": "Completed", "complete": "Completed",
    "cancelled": "Cancelled", "canceled": "Cancelled",
    "delivered": "Delivered", "shipped": "Shipped",
    "refunded": "Refunded", "refund": "Refunded", "refund - partial": "Refund - Partial",
    "pending": "Pending", "processing": "Processing",
}
ACCOUNT_STATUS_MAP = {"active": "Active", "deactivated": "Deactivated", "vacation": "Vacation",
                      "on vacation": "Vacation", "suspended": "Suspended"}

DATE_FORMATS = [
    "%Y-%m-%dT%H:%M:%SZ",   # 2024-11-24T17:52:54Z
    "%Y-%m-%d %H:%M:%S",    # 2024-01-24 20:02:32
    "%m/%d/%Y %H:%M",       # 10/07/2023 23:16
    "%m/%d/%Y",             # 09/06/2025
    "%m/%d/%y",             # 08/08/19
    "%Y-%m-%d",             # 2022-01-11
    "%d-%b-%Y",             # 25-Nov-2017
    "%B %d, %Y",            # April 22, 2018
]


def clean_text(series):
    # trim, collapse inner whitespace; sentinel words -> missing
    s = series.str.strip().str.replace(r"\s+", " ", regex=True)
    return s.mask(s.str.lower().isin(MISSING_SENTINELS))


def normalize_state(value):
    if pd.isna(value):
        return np.nan
    cleaned = str(value).strip().lower()
    if cleaned in US_STATE_TO_ABBR:
        return US_STATE_TO_ABBR[cleaned]
    return cleaned.upper() if cleaned.upper() in VALID_STATE_ABBRS else np.nan


def normalize_bool_flag(value):
    if pd.isna(value):
        return np.nan
    cleaned = str(value).strip().lower()
    return True if cleaned in {"y", "yes", "true", "1"} else False if cleaned in {"n", "no", "false", "0"} else np.nan


def parse_mixed_dates(series):
    # Try each known format explicitly, only on the rows still unparsed. Never let pandas guess:
    # a guessed format silently reads 03/04/2024 as 4 March in one row and 3 April in another.
    cleaned = series.str.strip()
    result = pd.Series(pd.NaT, index=series.index, dtype="datetime64[ns]")
    for fmt in DATE_FORMATS:
        mask = result.isna() & cleaned.notna()
        if not mask.any():
            break
        parsed = pd.to_datetime(cleaned[mask], format=fmt, errors="coerce")
        if getattr(parsed.dt, "tz", None) is not None:   # the ISO "...Z" format comes back tz-aware
            parsed = parsed.dt.tz_localize(None)
        result.loc[mask] = parsed
    return result


def normalize_category(series):
    return clean_text(series).str.title().fillna("Unknown")   # missing category is its own segment, not a dropped row

## Step 4: Clean the two tables (standardise + de-duplicate)

* De-duplication keeps the **latest ingested** version of each key (warehouses deliver rows *at least once*,
  and a refund re-sends the order with a new status).
* Missing values are classified by **mechanism** and kept as flags; nothing is imputed here (that happens in
  Step 8, *after* the split dates are known).
* A missing category becomes the explicit segment `Unknown`: the order still happened, and the `ALL` total
  must include it.

In [ ]:
sellers = sellers_raw.copy()
sellers["seller_id"] = sellers["seller_id"].str.strip()
sellers["row_ingested_at"] = pd.to_datetime(sellers["row_ingested_at"], errors="coerce")
sellers = (sellers.drop_duplicates()
           .sort_values("row_ingested_at").drop_duplicates(subset="seller_id", keep="last"))

sellers["signup_date"] = parse_mixed_dates(sellers["signup_date_raw"])
sellers["last_active_date"] = parse_mixed_dates(sellers["last_active_date_raw"])
sellers["primary_category"] = normalize_category(sellers["primary_category_raw"])
sellers["seller_type"] = clean_text(sellers["seller_type_raw"]).str.title()
sellers["signup_channel"] = clean_text(sellers["signup_channel_raw"]).str.title()
sellers["account_status"] = clean_text(sellers["account_status_raw"]).str.lower().map(ACCOUNT_STATUS_MAP)
sellers["state"] = sellers["state_raw"].apply(normalize_state)
sellers["is_star_seller"] = sellers["is_star_seller_raw"].apply(normalize_bool_flag)
sellers["zip_code"] = sellers["zip_code_raw"].str.strip().str.zfill(5)        # leading zeros were dropped upstream
sellers["email_domain"] = clean_text(sellers["email_domain"]).str.lower()

sellers_clean = sellers[["seller_id", "signup_date", "primary_category", "seller_type", "state",
                         "account_status", "is_star_seller", "signup_channel", "email_domain", "city",
                         "zip_code", "last_active_date", "row_ingested_at"]].reset_index(drop=True)
print(sellers_clean.shape)
sellers_clean.head(3)

In [ ]:
tx = transactions_raw.copy()
for id_col in ["order_id", "seller_id", "buyer_id"]:
    tx[id_col] = tx[id_col].str.strip()
tx["row_ingested_at"] = pd.to_datetime(tx["row_ingested_at"], errors="coerce")
tx = tx.drop_duplicates().sort_values("row_ingested_at").drop_duplicates(subset="order_id", keep="last")

tx["order_datetime"] = parse_mixed_dates(tx["order_datetime_raw"])
tx["category"] = normalize_category(tx["category_raw"])
tx["order_status"] = clean_text(tx["order_status_raw"]).str.lower().map(ORDER_STATUS_MAP)
tx["currency"] = clean_text(tx["currency"]).str.upper()
tx["buyer_state"] = tx["buyer_state_raw"].apply(normalize_state)
tx["is_gift"] = tx["is_gift_flag"].apply(normalize_bool_flag)
for col in ["payment_method", "device_type", "source_system", "coupon_code"]:
    tx[col] = clean_text(tx[col])
tx["item_price_usd"] = pd.to_numeric(tx["item_price_usd"], errors="coerce")
tx["shipping_fee_usd"] = pd.to_numeric(tx["shipping_fee_usd"], errors="coerce")
tx["quantity"] = pd.to_numeric(tx["quantity"], errors="coerce")

# missing-value mechanisms: "no coupon" is a real value; "price missing" is a pipeline drop-out -> flag, impute later
tx["has_coupon"] = tx["coupon_code"].notna()
tx["price_missing"] = tx["item_price_usd"].isna()
tx["shipping_missing"] = tx["shipping_fee_usd"].isna()

transactions = tx[["order_id", "seller_id", "buyer_id", "order_datetime", "source_system", "category",
                   "item_price_usd", "quantity", "shipping_fee_usd", "currency", "buyer_state",
                   "buyer_country", "payment_method", "order_status", "device_type", "is_gift", "has_coupon",
                   "price_missing", "shipping_missing", "row_ingested_at"]].reset_index(drop=True)
print(transactions.shape)
print(transactions["order_status"].value_counts(dropna=False).to_string())

## Step 5: Quarantine invalid rows (with a reason)

Rows that fail a validity rule are **set aside with a reason**, not deleted and not kept:
deleting hides upstream bugs, keeping corrupts the model. Each row gets the *first* rule it fails, so the
report is easy to act on. If more than `max_quarantine_rate` of a table fails, the run stops - that is a
broken upstream system, not noise.

Note what is **not** quarantined: a missing price. The order still counts as a transaction; only GMV
needs the price, and it is imputed in Step 8.

In [ ]:
def quarantine(df, checks, table):
    # checks = [(reason, boolean mask of BAD rows)], evaluated in order; first failure wins.
    reason = pd.Series(pd.NA, index=df.index, dtype="string")
    for name, bad in checks:
        bad = bad.fillna(False).astype(bool)
        reason = reason.mask(reason.isna() & bad, name)
    rejected = df[reason.notna()].assign(quarantine_reason=reason[reason.notna()])
    rejected.to_csv(cfg.out_dir / f"quarantine_{table}.csv", index=False)
    rate = len(rejected) / len(df)
    print(f"{table}: quarantined {len(rejected):,} of {len(df):,} rows ({rate:.2%})")
    if len(rejected):
        print(rejected["quarantine_reason"].value_counts().to_string())
    if rate > cfg.max_quarantine_rate:
        raise RuntimeError(f"{table}: quarantine rate {rate:.2%} > {cfg.max_quarantine_rate:.0%} - upstream is broken")
    return df[reason.isna()].reset_index(drop=True)


tx_snapshot = transactions["row_ingested_at"].max()          # when the extract was taken
sellers_snapshot = sellers_clean["row_ingested_at"].max()
print("transactions snapshot:", tx_snapshot, "| sellers snapshot:", sellers_snapshot)

transactions = quarantine(transactions, [
    ("unparseable order_datetime", transactions["order_datetime"].isna()),
    ("order after extract snapshot", transactions["order_datetime"] > tx_snapshot),
    ("unknown order_status", transactions["order_status"].isna()),
    ("not a transaction status", ~transactions["order_status"].isin(cfg.transaction_statuses)),
    (f"currency not {cfg.currency}", transactions["currency"] != cfg.currency),
    ("quantity missing or <= 0", transactions["quantity"].isna() | (transactions["quantity"] <= 0)),
    ("negative price or shipping", (transactions["item_price_usd"] < 0) | (transactions["shipping_fee_usd"] < 0)),
], "transactions")

sellers_clean = quarantine(sellers_clean, [
    ("unparseable signup_date", sellers_clean["signup_date"].isna()),
    ("signup after extract snapshot", sellers_clean["signup_date"] > sellers_snapshot),
], "sellers")

## Step 6: Data maturity - find the last *complete* day

The newest days are always missing late arrivals (offline apps, batch retries, nightly loads).
`row_ingested_at - order_datetime` tells us how long rows take to arrive. We measure its 99th percentile on
**mature** events only (30-120 days old - newer events' stragglers have not arrived yet, so including them
would make the delay look shorter than it is), and cut off every day that cannot be complete yet.

The production forecast origin is this **last complete day**, not "today".

In [ ]:
delay = transactions["row_ingested_at"] - transactions["order_datetime"]
age_days = (tx_snapshot - transactions["order_datetime"]).dt.days
mature = delay[(age_days >= 30) & (age_days <= 120) & (delay >= pd.Timedelta(0))]
required_delay = mature.quantile(cfg.maturity_quantile)

orders_cutoff = (tx_snapshot - required_delay).normalize() - pd.Timedelta(days=1)
sellers_cutoff = sellers_snapshot.normalize() - pd.Timedelta(days=1)   # full-table snapshot: yesterday is complete

print(f"ingestion delay p{cfg.maturity_quantile:.0%} = {required_delay}  (median {mature.median()})")
print(f"last complete day  orders: {orders_cutoff.date()} | new sellers: {sellers_cutoff.date()}")
n_immature = (transactions["order_datetime"] > orders_cutoff + pd.Timedelta(days=1)).sum()
print(f"rows after the orders cut-off (kept in the clean table, excluded from the panel): {n_immature:,}")

## Step 7: Fix the split dates *before* fitting anything

Time series are never split randomly. From the cut-off we fix, once, for each target:

| `split` | Rows | Used for |
|---|---|---|
| `train` | oldest origins | fitting; rolling-origin backtest folds live inside it |
| `purged` | origins whose target window would reach into the test period (+ embargo) | nothing - removed |
| `test` | the most recent origins whose longest-horizon target is complete, and everything after | evaluated **once**, at the very end |
| `inference` | origin = last complete day | the production forecast row (targets unknown) |

Every fitted statistic below (imputation medians, outlier caps) uses rows dated **≤ `train_origin_end`** only.

In [ ]:
def split_plan(cutoff):
    max_h = max(cfg.horizons)
    test_origin_start = cutoff - pd.Timedelta(days=max_h + cfg.test_days - 1)
    train_origin_end = test_origin_start - pd.Timedelta(days=1 + cfg.embargo_days + max_h)
    return {"cutoff": cutoff, "test_origin_start": test_origin_start, "train_origin_end": train_origin_end}


plans = {"orders": split_plan(orders_cutoff), "new_sellers": split_plan(sellers_cutoff)}
for name, plan in plans.items():
    print(name, {k: v.date() for k, v in plan.items()})

## Step 8: Train-fitted imputation and outlier caps (row level)

Same techniques as v1 (per-category median, per-category winsorising, robust MAD flag) with one change
that makes them leakage-safe: **fit on the training period, apply everywhere**. Interpolation with
`limit_direction="both"`, KNN and KMeans imputers were dropped - they fill a gap using rows from *after* it.

Raw values are never overwritten: `item_price_usd` stays as delivered, `item_price_usd_filled` is the
modelling version, and `order_total_usd` is computed **after** imputation (v1 computed it before, so GMV
ignored every imputed order).

In [ ]:
train_rows = transactions["order_datetime"] <= plans["orders"]["train_origin_end"]
fit = transactions[train_rows]

# 1) impute missing prices/shipping with the TRAINING-period median of the row's category
for col in ["item_price_usd", "shipping_fee_usd"]:
    medians = fit.groupby("category")[col].median()
    transactions[f"{col}_filled"] = transactions[col].fillna(transactions["category"].map(medians)).fillna(fit[col].median())

# 2) flag outliers with a robust per-category z-score whose median/MAD come from the training period
stats = fit.groupby("category")["item_price_usd"].agg(median="median", mad=lambda s: (s - s.median()).abs().median())
robust_z = ((transactions["item_price_usd"] - transactions["category"].map(stats["median"]))
            / (transactions["category"].map(stats["mad"]).replace(0, np.nan) * 1.4826))
transactions["is_price_outlier"] = (robust_z.abs() > 3.5).fillna(False)

# 3) winsorise with training-period per-category caps (modelling column; raw column untouched)
caps = fit.groupby("category")["item_price_usd"].quantile([0.005, 0.995]).unstack()
transactions["item_price_usd_w"] = transactions["item_price_usd_filled"].clip(
    lower=transactions["category"].map(caps[0.005]), upper=transactions["category"].map(caps[0.995]))

transactions["order_total_usd"] = (transactions["item_price_usd_filled"] * transactions["quantity"]
                                   + transactions["shipping_fee_usd_filled"])
print(f"fitted on {train_rows.sum():,} training rows; price outliers flagged: {transactions['is_price_outlier'].sum():,}")
print(transactions[["item_price_usd", "item_price_usd_filled", "item_price_usd_w", "order_total_usd"]].describe().round(2))

## Step 9: Daily panel on a complete calendar spine

One row per **(segment, day)** from each segment's first day to the cut-off. A day with no rows is an
explicit **0** - a missing day would silently shift every lag computed later. The `ALL` segment is the total,
and a gate checks the categories still add up to it (hierarchical consistency).

* `orders` = every row with a transaction status; `gmv_usd` = `order_total_usd` of revenue-bearing statuses only.
* `new_sellers` = sign-ups per day by primary category.

In [ ]:
def build_daily_panel(df, date_col, segment_col, value_cols, end):
    # Dense (segment x day) grid up to `end`, explicit zeros from each segment's first day; 'ALL' = sum over segments.
    df = df[df[date_col] <= end + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)]
    daily = (df.assign(date=df[date_col].dt.floor("D"), segment=df[segment_col])
               .groupby(["segment", "date"])[value_cols].sum())
    total = daily.groupby("date").sum().assign(segment="ALL").set_index("segment", append=True).swaplevel()
    frames = []
    for seg, g in pd.concat([daily, total]).groupby(level="segment"):
        g = g.droplevel("segment")
        g = g.reindex(pd.date_range(g.index.min(), end, freq="D"), fill_value=0)
        frames.append(g.rename_axis("date").reset_index().assign(segment=seg))
    return (pd.concat(frames)[["segment", "date"] + value_cols]
              .sort_values(["segment", "date"]).reset_index(drop=True))


transactions["orders"] = 1
transactions["gmv_usd"] = transactions["order_total_usd"].where(transactions["order_status"].isin(cfg.gmv_statuses), 0.0)
orders_panel = build_daily_panel(transactions, "order_datetime", "category", ["orders", "gmv_usd"], orders_cutoff)

sellers_clean["new_sellers"] = 1
sellers_panel = build_daily_panel(sellers_clean, "signup_date", "primary_category", ["new_sellers"], sellers_cutoff)

for name, panel, col in [("orders", orders_panel, "orders"), ("new_sellers", sellers_panel, "new_sellers")]:
    total = panel[panel["segment"] == "ALL"].set_index("date")[col]
    parts = panel[panel["segment"] != "ALL"].groupby("date")[col].sum().reindex(total.index, fill_value=0)
    # freshness: is the newest "complete" day really complete? A failed extract job shows up as a sudden drop
    # against the previous week (the ingestion-delay analysis cannot see a job that never ran)
    last, baseline = total.iloc[-1], total.iloc[-8:-1].median()
    flag = "" if last >= 0.5 * baseline else "  <-- WARNING: newest day looks half-loaded, check the extract job"
    print(f"{name}: {panel.shape[0]:,} rows, {panel['segment'].nunique()} segments, "
          f"{panel['date'].min().date()} -> {panel['date'].max().date()} | reconciles: {bool((total == parts).all())} "
          f"| newest day {last:.0f} vs previous-week median {baseline:.0f}{flag}")
orders_panel.tail(3)

## Step 10: Calendar - known-future features

Holidays, the Cyber 5 (Thanksgiving to Cyber Monday), the gifting run-ups (Valentine's, Mother's Day,
Father's Day, Christmas - buyers order *ahead* because items are handmade) and the post-Christmas trough are
known **years ahead**, so a row may legitimately say *"the next 30 days contain Cyber Monday"*. Every date
comes from a rule (Thanksgiving = 4th Thursday of November), so the calendar extends to any year. It is built
past the cut-off so the target windows of the newest origins are covered. Seasonality is encoded with sin/cos
waves instead of one-hot buckets, so a model knows Sunday sits next to Monday.

In [ ]:
def nth_weekday(year, month, weekday, n):
    first = pd.Timestamp(year, month, 1)                      # Monday=0 ... Sunday=6
    return first + pd.Timedelta(days=(weekday - first.weekday()) % 7 + 7 * (n - 1))


def build_calendar(start, end):
    cal = pd.DataFrame({"date": pd.date_range(start, end, freq="D")})
    cal["is_us_holiday"] = cal["date"].isin(USFederalHolidayCalendar().holidays(start, end)).astype(int)
    cyber5, gifting, trough = [], [], []
    for year in range(start.year, end.year + 1):
        thanksgiving = nth_weekday(year, 11, 3, 4)                          # 4th Thursday of November
        cyber5 += list(pd.date_range(thanksgiving, thanksgiving + pd.Timedelta(days=4)))
        # buyers order AHEAD of a gifting occasion (handmade / personalised), so each window sits before it
        for occasion, lead_days in [(pd.Timestamp(year, 2, 14), 14), (nth_weekday(year, 5, 6, 2), 14),   # Valentine's, Mother's Day
                                    (nth_weekday(year, 6, 6, 3), 14), (pd.Timestamp(year, 12, 25), 40)]:  # Father's Day, Christmas
            gifting += list(pd.date_range(occasion - pd.Timedelta(days=lead_days), occasion))
        trough += list(pd.date_range(pd.Timestamp(year, 12, 26), pd.Timestamp(year + 1, 1, 5)))        # post-Christmas lull
    cal["is_cyber5"] = cal["date"].isin(cyber5).astype(int)
    cal["is_gifting_window"] = cal["date"].isin(gifting).astype(int)
    cal["is_post_holiday_trough"] = cal["date"].isin(trough).astype(int)
    to_xmas = (pd.to_datetime(cal["date"].dt.year.astype(str) + "-12-25") - cal["date"]).dt.days
    cal["days_to_christmas"] = to_xmas.where(to_xmas >= 0, to_xmas + 365)
    for k in (1, 2):                                          # weekly cycle
        cal[f"dow_sin_{k}"] = np.sin(2 * np.pi * k * cal["date"].dt.dayofweek / 7)
        cal[f"dow_cos_{k}"] = np.cos(2 * np.pi * k * cal["date"].dt.dayofweek / 7)
    for k in (1, 2, 3):                                       # yearly cycle
        cal[f"doy_sin_{k}"] = np.sin(2 * np.pi * k * cal["date"].dt.dayofyear / 365.25)
        cal[f"doy_cos_{k}"] = np.cos(2 * np.pi * k * cal["date"].dt.dayofyear / 365.25)
    return cal


def add_calendar_features(panel, calendar):
    # 1) the origin day's own calendar; 2) how many special days fall inside each target window (t, t+H]
    out = panel.merge(calendar, on="date", how="left")
    for flag in ["is_us_holiday", "is_cyber5", "is_gifting_window"]:
        running = calendar.set_index("date")[flag].cumsum()
        for H in cfg.horizons:
            out[f"fut_h{H}_{flag[3:]}_days"] = (running.reindex(out["date"] + pd.Timedelta(days=H)).to_numpy()
                                               - running.reindex(out["date"]).to_numpy())
    return out


calendar_start = min(orders_panel["date"].min(), sellers_panel["date"].min())
calendar_end = max(orders_cutoff, sellers_cutoff) + pd.Timedelta(days=max(cfg.horizons) + 7)
calendar = build_calendar(calendar_start, calendar_end)
known_high_variance = calendar.loc[(calendar[["is_cyber5", "is_gifting_window", "is_post_holiday_trough"]].sum(axis=1) > 0), "date"]
print(f"calendar {calendar_start.date()} -> {calendar_end.date()} ({len(calendar):,} days); "
      f"cyber-5 days: {calendar['is_cyber5'].sum()}, gifting-window days: {calendar['is_gifting_window'].sum()}, "
      f"holidays: {calendar['is_us_holiday'].sum()}")

## Step 11: Anomaly screen - separate *label* and *feature* series

Each day is compared with the median of the **same weekday over the previous 4 weeks** (backward-looking
only), on the square-root scale so a spike of +30 on a 30/day series and +300 on a 3,000/day series look
alike. Days beyond `anomaly_z` robust standard deviations are flagged.

Two copies of every count series come out of this:

* `<series>` (the **label**): what really happened - targets are built from it, because a real outage *did*
  happen and the model must be judged on it.
* `<series>_feature`: anomaly days replaced by the baseline, so one freak day does not distort lags and
  rolling means for weeks.

Days inside the calendar's known high-variance windows (gifting run-ups, Cyber 5, the post-Christmas trough)
are **never flagged**: a Cyber Monday spike is not an anomaly, and replacing it would erase next year's
`lag_364` signal. (The first version of this screen flagged 91 days on `ALL` - almost all of them Q4.)

With an incident log you would also distinguish *data loss* (fix the label too) from *real disruption*
(keep the label). Without one, the flagged days are written to `anomaly_report.csv` for human review.

In [ ]:
def screen_anomalies(panel, col):
    # returns panel + f"{col}_feature" + f"{col}_is_anomaly"; everything here looks backwards only
    g = panel.groupby("segment")[col]
    sqrt = lambda s: 2 * np.sqrt(s + 3 / 8)                                     # variance-stabilising (Anscombe)
    baseline = pd.concat([g.shift(k) for k in (7, 14, 21, 28)], axis=1).median(axis=1)
    resid = sqrt(panel[col]) - sqrt(baseline)
    scale = (resid.abs().groupby(panel["segment"])
             .transform(lambda s: s.shift(1).rolling(91, min_periods=28).median()) * 1.4826).clip(lower=0.5)
    z = (resid / scale).mask(panel["date"].isin(known_high_variance), 0.0)   # a Cyber Monday spike is not an anomaly
    out = panel.copy()
    out[f"{col}_is_anomaly"] = (z.abs() > cfg.anomaly_z).fillna(False).astype(int)
    out[f"{col}_feature"] = panel[col].where(out[f"{col}_is_anomaly"] == 0, baseline.round())
    out["_z"] = z
    return out


def anomaly_report(panel, col):
    flagged = panel[panel[f"{col}_is_anomaly"] == 1]
    return pd.DataFrame({"series": col, "segment": flagged["segment"], "date": flagged["date"].dt.date,
                         "actual": flagged[col], "baseline_used": flagged[f"{col}_feature"], "robust_z": flagged["_z"].round(2)})


reports = []
for name, panel, col in [("orders", orders_panel, "orders"), ("new_sellers", sellers_panel, "new_sellers")]:
    screened = screen_anomalies(panel, col)
    reports.append(anomaly_report(screened, col))
    print(f"{name}: {screened[f'{col}_is_anomaly'].sum()} anomaly day(s) flagged across {screened['segment'].nunique()} segments")
anomalies = pd.concat(reports)
anomalies.to_csv(cfg.out_dir / "anomaly_report.csv", index=False)
anomalies[anomalies["segment"] == "ALL"].head(10)

## Step 12: History features, then targets - the point-in-time rule made concrete

Row = origin *t*. For each series, every feature uses days **≤ t**:

| Feature | Definition |
|---|---|
| `lag_k` | value on day *t-k* (`lag_0` = the origin day itself, the last fully observed day) |
| `roll_mean_w`, `roll_std_28` | window of *w* days **ending on t** (inclusive) |
| `ewm_03` | exponentially weighted mean up to *t* |
| `momentum_7_28` | `roll_mean_7 / roll_mean_28` - is the last week above its monthly run-rate? |
| `yoy_28` | `roll_mean_28 / roll_mean_28 one year earlier` - growth, weekday-aligned (364 = 52 weeks) |
| `aov_28` | GMV / orders over the last 28 days (replaces v1's same-day ratio) |

Targets: `y_h{H} = label(t+1) + ... + label(t+H)`, `NaN` when the window runs past the last complete day.
Direct multi-horizon targets avoid piling up the error of 90 chained one-day forecasts. For tree models, which
cannot extrapolate a trend, `y_h{H}_scaled = y_h{H} / (H * roll_mean_28)` stays near 1 as volume grows.

In [ ]:
def add_history_features(panel, source_col, name):
    g = panel.groupby("segment")[source_col]
    f = {}
    for k in cfg.lags:
        f[f"{name}_lag_{k}"] = g.shift(k)
    for w in cfg.roll_windows:
        f[f"{name}_roll_mean_{w}"] = g.transform(lambda s: s.rolling(w, min_periods=w).mean())
    f[f"{name}_roll_std_28"] = g.transform(lambda s: s.rolling(28, min_periods=28).std())
    f[f"{name}_ewm_03"] = g.transform(lambda s: s.ewm(alpha=0.3).mean())
    f = pd.DataFrame(f, index=panel.index)
    f[f"{name}_momentum_7_28"] = f[f"{name}_roll_mean_7"] / f[f"{name}_roll_mean_28"].replace(0, np.nan)
    f[f"{name}_yoy_28"] = f[f"{name}_roll_mean_28"] / f.groupby(panel["segment"])[f"{name}_roll_mean_28"].shift(364).replace(0, np.nan)
    return pd.concat([panel, f], axis=1)


def add_targets(panel, label_col):
    g = panel.groupby("segment")[label_col]
    for H in cfg.horizons:
        # rolling(H).sum() at row t+H holds label[t+1 .. t+H]; shift(-H) moves it back to row t
        panel[f"y_h{H}"] = g.transform(lambda s: s.rolling(H, min_periods=H).sum().shift(-H))
        panel[f"y_h{H}_scaled"] = panel[f"y_h{H}"] / (H * panel[f"{label_col}_roll_mean_28"].replace(0, np.nan))
    return panel


def build_model_table(panel, target, extra_cols=()):
    # panel -> anomaly screen -> history features (on the cleaned series) -> calendar -> targets (on the label)
    table = screen_anomalies(panel, target)
    table = add_history_features(table, f"{target}_feature", target)
    for col in extra_cols:
        table = screen_anomalies(table, col)
        table = add_history_features(table, f"{col}_feature", col)
    if "gmv_usd" in extra_cols:
        table["aov_28"] = table["gmv_usd_roll_mean_28"] / table["orders_roll_mean_28"].replace(0, np.nan)
    table = add_calendar_features(table, calendar)
    return add_targets(table, target)


orders_table = build_model_table(orders_panel, "orders", extra_cols=("gmv_usd",))
sellers_table = build_model_table(sellers_panel, "new_sellers")
print("orders table", orders_table.shape, "| new_sellers table", sellers_table.shape)
orders_table[orders_table["segment"] == "ALL"][["date", "orders", "orders_lag_0", "orders_lag_7", "orders_roll_mean_28",
                                                 "fut_h30_cyber5_days", "y_h7", "y_h30"]].tail(8)

## Step 13: Assign the split, drop the burn-in

Each origin gets the role fixed in Step 7. Additionally:

* **burn-in**: a segment's first 364 + 28 origins are dropped from training - `yoy_28` needs a 28-day mean from
  one year earlier, and a model cannot learn from a feature that is always missing.
* `sample_weight`: recent rows weigh more (half-life 2 years) - marketplaces drift, and 2023 behaviour
  should not count as much as last quarter's. Rows whose target window contains a flagged anomaly day get
  half weight. Models that accept weights can use it; others ignore it.

In [ ]:
META_PREFIXES = ("y_h",)
META_COLUMNS = {"segment", "date", "split", "sample_weight", "_z"}


def feature_columns(table):
    return [c for c in table.columns
            if c not in META_COLUMNS and not c.startswith(META_PREFIXES)
            and not c.endswith(("_feature", "_is_anomaly"))
            and c not in ("orders", "gmv_usd", "new_sellers")]


def assign_split(table, target, plan):
    date, max_h = table["date"], max(cfg.horizons)
    split = np.select(
        [date == plan["cutoff"], date >= plan["test_origin_start"], date > plan["train_origin_end"]],
        ["inference", "test", "purged"], default="train")
    # yoy_28 needs a 28-day mean from one year earlier, so a segment's first 364 + 28 origins cannot be trained on
    burn_in_days = max(cfg.lags + cfg.roll_windows) + 28
    first_day = table.groupby("segment")["date"].transform("min")
    burn_in = (date < first_day + pd.Timedelta(days=burn_in_days)) & (split != "inference")
    table = table.assign(split=np.where(burn_in, "burn_in", split))

    age_days = (plan["cutoff"] - date).dt.days
    anomaly_in_window = (table.groupby("segment")[f"{target}_is_anomaly"]
                         .transform(lambda s: s.rolling(max_h, min_periods=1).sum().shift(-max_h)).fillna(0))
    table["sample_weight"] = 0.5 ** (age_days / 730) * np.where(anomaly_in_window > 0, 0.5, 1.0)
    return table.drop(columns="_z")


orders_table = assign_split(orders_table, "orders", plans["orders"])
sellers_table = assign_split(sellers_table, "new_sellers", plans["new_sellers"])

for name, table in [("orders", orders_table), ("new_sellers", sellers_table)]:
    print(f"\n{name}: {len(feature_columns(table))} features")
    print(table[table["segment"] == "ALL"].groupby("split")["date"].agg(["min", "max", "count"]).to_string())

## Step 14: Quality gates + the automatic leakage test

**Table checks:** one row per (segment, day) with no gaps; the panel ends on the cut-off; categories add up to
`ALL`; every `train`/`test` row has all targets; the inference row has none; no infinite values.

**Truncation leakage test:** for a few origins *t*, throw away all data after *t*, rebuild the whole feature
pipeline, and check that the feature row for *t* is **identical** to the one from the full build. A centred
rolling window, a stray `shift(-1)`, or an anomaly screen that peeks ahead all fail this test - and code review
rarely catches them.

In [ ]:
def leakage_truncation_test(panel, target, table, n_origins=3, extra_cols=()):
    features = feature_columns(table)
    origins = table.loc[(table["split"] == "train") & (table["segment"] == "ALL"), "date"]
    origins = origins.iloc[np.linspace(0, len(origins) - 1, n_origins).astype(int)]
    for origin in origins:
        rebuilt = build_model_table(panel[panel["date"] <= origin], target, extra_cols)
        full_row = table[table["date"] == origin].set_index("segment")[features]
        trunc_row = rebuilt[rebuilt["date"] == origin].set_index("segment")[features].reindex(full_row.index)
        same = np.isclose(full_row.to_numpy(float), trunc_row.to_numpy(float), equal_nan=True)
        if not same.all():
            leaky = sorted(set(np.array(features)[np.where(~same)[1]]))
            raise RuntimeError(f"LEAKAGE at origin {origin.date()}: features change when future data is removed: {leaky}")
    return f"{len(features)} features identical at {n_origins} origins"


def table_gates(table, target, plan, name):
    by_seg = table.groupby("segment")["date"]
    feats = feature_columns(table)
    y_cols = [f"y_h{H}" for H in cfg.horizons]
    total = table[table["segment"] == "ALL"].set_index("date")[target]
    parts = table[table["segment"] != "ALL"].groupby("date")[target].sum().reindex(total.index, fill_value=0)
    # a target is known exactly when its whole window (t, t+H] lies on or before the cut-off
    test = table[table["split"] == "test"]
    test_targets_ok = all(((test["date"] + pd.Timedelta(days=H) <= plan["cutoff"]) == test[f"y_h{H}"].notna()).all()
                          for H in cfg.horizons)
    train_missing = table.loc[table["split"] == "train", feats].isna().mean().max()
    return [
        (f"{name}: (segment, date) unique", not table.duplicated(["segment", "date"]).any(), ""),
        (f"{name}: no calendar gaps", bool((by_seg.count() == (by_seg.max() - by_seg.min()).dt.days + 1).all()), ""),
        (f"{name}: ends on the last complete day", table["date"].max() == plan["cutoff"], str(plan["cutoff"].date())),
        (f"{name}: categories add up to ALL", bool((total == parts).all()), ""),
        (f"{name}: train rows have every target", table.loc[table["split"] == "train", y_cols].notna().all().all(), ""),
        (f"{name}: test targets missing only when the window passes the cut-off", test_targets_ok, ""),
        (f"{name}: inference rows have no target", table.loc[table["split"] == "inference", y_cols].isna().all().all(), ""),
        (f"{name}: exactly one inference row per segment", (table[table["split"] == "inference"].groupby("segment").size() == 1).all(), ""),
        (f"{name}: no infinite feature values", not np.isinf(table[feats].to_numpy(float)).any(), ""),
        (f"{name}: no feature is missing in > 10% of train rows", train_missing < 0.10, f"max {train_missing:.2%}"),
    ]


gates = table_gates(orders_table, "orders", plans["orders"], "orders") + table_gates(sellers_table, "new_sellers", plans["new_sellers"], "new_sellers")
gates += [("orders: truncation leakage test", True, leakage_truncation_test(orders_panel, "orders", orders_table, extra_cols=("gmv_usd",))),
          ("new_sellers: truncation leakage test", True, leakage_truncation_test(sellers_panel, "new_sellers", sellers_table))]
run_gates(gates, "POST-CLEAN")

## Step 15: Save outputs + run manifest

Everything a model needs, plus a manifest that ties the outputs to the exact settings, cut-offs, row counts
and library versions that produced them. `feature_dictionary.csv` is generated from the column names, so it
can never drift out of date.

In [ ]:
def describe_feature(col):
    rules = [("_lag_", "value {n} day(s) before the origin (0 = the origin day)"), ("_roll_mean_", "mean of the {n} days ending on the origin"),
             ("_roll_std_", "std of the {n} days ending on the origin"), ("_ewm_", "exponentially weighted mean up to the origin"),
             ("_momentum_", "last 7 days / last 28 days"), ("_yoy_", "last 28 days / same 28 days one year earlier"),
             ("aov_", "GMV per order over the last 28 days"), ("fut_h", "count of such days inside the next H days (known in advance)"),
             ("dow_", "weekly cycle (sin/cos)"), ("doy_", "yearly cycle (sin/cos)"), ("is_", "calendar flag for the origin day"),
             ("days_to_", "days until the event")]
    for key, text in rules:
        if key in col:
            n = col.rsplit("_", 1)[-1]
            return text.format(n=n)
    return ""


outputs = {}
for name, df in [("sellers_clean", sellers_clean.drop(columns="new_sellers")), ("transactions_clean", transactions.drop(columns=["orders", "gmv_usd"])),
                 ("orders_model_table", orders_table), ("new_sellers_model_table", sellers_table), ("calendar", calendar)]:
    path = cfg.out_dir / f"{name}.csv"
    df.to_csv(path, index=False)
    outputs[name] = {"path": str(path), "rows": int(len(df)), "cols": int(df.shape[1])}
    print(f"saved {path}  {len(df):,} rows x {df.shape[1]} cols")

dictionary = pd.DataFrame([{"table": t, "feature": c, "description": describe_feature(c)}
                           for t, tbl in [("orders", orders_table), ("new_sellers", sellers_table)] for c in feature_columns(tbl)])
dictionary.to_csv(cfg.out_dir / "feature_dictionary.csv", index=False)

manifest = {
    "run_at": datetime.now().isoformat(timespec="seconds"),
    "config": asdict(cfg),
    "snapshots": {"transactions": str(tx_snapshot), "sellers": str(sellers_snapshot)},
    "ingestion_delay_p99": str(required_delay),
    "split_plans": {k: {kk: str(vv.date()) for kk, vv in v.items()} for k, v in plans.items()},
    "split_counts": {k: t[t["segment"] == "ALL"]["split"].value_counts().to_dict() for k, t in [("orders", orders_table), ("new_sellers", sellers_table)]},
    "anomaly_days_flagged": int(len(anomalies)),
    "outputs": outputs,
    "versions": {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__},
}
(cfg.out_dir / "run_manifest.json").write_text(json.dumps(manifest, indent=2, default=str))
print("\nmanifest written:", cfg.out_dir / "run_manifest.json")

## How to use the output (and what comes next)

Each model table has one row per **(segment, origin)**: `segment`, `date`, the raw series, `split`,
`sample_weight`, the features (`feature_columns(table)`), and the targets `y_h7 ... y_h90` (+ `_scaled`).

```python
table = orders_table[orders_table["segment"] == "ALL"]
X_cols = feature_columns(table)
train, test = table[table["split"] == "train"], table[table["split"] == "test"]
# one model per horizon:  fit(train[X_cols], train["y_h30"], sample_weight=train["sample_weight"])
```

**Next steps, one at a time (each is a small, separate change):**

1. **Baselines first** - seasonal naive (`lag_6`... same weekday last week × H) and "same window last year × yoy".
   A model is only worth shipping if it beats these.
2. **Rolling-origin backtest** inside `train` (expanding window, step = 30 days), metrics per horizon
   (WAPE, MAE), *never* a shuffled K-fold.
3. **Feature selection inside the folds** - the correlation/VIF/noise-probe/RFE logic from v1 is fine,
   but it must see training folds only.
4. **One direct model per horizon** (LightGBM on `y_h{H}_scaled`, Poisson/Tweedie objective on `y_h{H}`),
   then the `test` split once, at the very end.
5. **Per-horizon purging** (this notebook purges conservatively with the longest horizon for every table).
6. **Incident log + marketing calendar** as inputs: data-loss days get fixed labels; planned campaigns become
   known-future features (only with their *planned-at* date, or they leak).